# Granted planning permissions → private housing starts (England, 1979Q2–2025Q4)

Distributed-lag model of England private-enterprise starts on granted planning
permissions, following the structural-relationships specification in Somerville
(2001, eq. 1, p.166) and the same selection protocol used for the
starts-to-completions bridge in §3.5.2: integration-order test → lag sweep →
own-lag sweep → HAC estimation → diagnostics → robustness.

**References used below** (BibTeX keys as they should appear in the dissertation
bibliography — note there is no `.bib` file in this repository, so the keys are
recorded here rather than verified):

- `somerville2001permits` — Somerville, C.T. (2001), *Permits, Starts, and Completions: Structural Relationships Versus Real Options*, Real Estate Economics 29(1), 161–190.
- `ball2011planning` — Ball, M. (2011), *UK Planning Controls and the Market Responsiveness of Housing Supply*, Urban Studies 48(2), 349–362.
- `payne2019landsupply` — Payne, S., Serin, B., James, G. & Adams, D. (2019), *How Does the Land Supply System Affect the Business of UK Speculative Housebuilding?*, UK Collaborative Centre for Housing Evidence.
- `ballcheshirehilberyu2024` — Ball, Cheshire, Hilber & Yu (2024), build-out lag paper (CEP).
- `letwin2018` — Letwin, O. (2018), *Independent Review of Build Out: Final Report*.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox, acorr_breusch_godfrey

warnings.filterwarnings("ignore")

ROOT = "../.."
RAW = f"{ROOT}/data/raw"
PM = f"{ROOT}/data/python_master"
OUT = f"{ROOT}/data/outputs/planning"

import os
os.makedirs(OUT, exist_ok=True)

HAC_LAGS = 4          # matches the completions bridge and the ARDL/VECM chapters
KMAX = 12             # statutory 3-year commencement condition (ballcheshirehilberyu2024)
PMAX = 4              # own-lag sweep range, same as the bridge protocol

## Data construction

Two changes to how the permissions series was built previously, both recorded
here rather than buried:

1. **Residential permissions only.** The earlier version used
   `Total granted; major total (all)`, which aggregates major permissions across
   *all* development types — offices, R&D, industry, warehousing, retail,
   traveller caravan pitches and "all other developments" — not just dwellings.
   Regressing housing starts on a series dominated by non-residential consents
   is a construction error rather than a specification choice, so the headline
   series here is `major dwellings (all) + minor dwellings (all)`. The original
   series is retained and reported in the robustness section (§5) so the change
   is visible and reversible.
2. **Units.** PS2 counts *decisions*, not dwellings. A coefficient on granted
   permissions is therefore "dwellings started per residential permission
   granted", not a unit-for-unit conversion rate. Majors are 10+ dwelling
   schemes and minors 1–9, so the mix behind one "permission" shifts over the
   sample — an interpretive caveat carried through to §6.

Starts are England private-enterprise starts from `england_master.csv` (the
same series the VECM/ARDL chapters use).

In [ ]:
ps2 = pd.read_csv(f"{RAW}/planning_applications/PS2_data_-_open_data_table__202512_.csv",
                  encoding="cp1252", skiprows=2, low_memory=False)
ps2["Quarter"] = pd.PeriodIndex(ps2["Quarter"].str.replace(" ", ""), freq="Q")

PS2_COLS = {
    "major_dw":  "Total granted; major dwellings (all)",
    "minor_dw":  "Total granted; minor dwellings (all)",
    "major_all": "Total granted; major total (all)",   # previous headline, kept for §5
}
for name, col in PS2_COLS.items():
    ps2[name] = pd.to_numeric(ps2[col], errors="coerce")

# England total: PS2 is one row per LPA per quarter
gr = ps2.groupby("Quarter")[list(PS2_COLS)].sum(min_count=1)
gr["granted"] = gr["major_dw"] + gr["minor_dw"]

master = pd.read_csv(f"{PM}/england_master.csv", index_col=0)
master.index = pd.PeriodIndex(master.index, freq="Q")

df = pd.concat([master[["starts", "hprice", "gdp_def", "vol"]],
                gr[["granted", "major_dw", "minor_dw", "major_all"]]], axis=1)
df = df.dropna(subset=["starts", "granted"])

gr.reset_index().to_csv(f"{PM}/planning/plan_granted.csv", index=False)

print(f"sample: {df.index[0]} - {df.index[-1]}  (n = {len(df)})")
df[["starts", "granted", "major_dw", "minor_dw", "major_all"]].describe().round(0)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
df["starts"].plot(ax=ax[0], color="#2c7bb6")
ax[0].set_ylabel("Private starts (dwellings)")
df["granted"].plot(ax=ax[1], color="#d7191c", label="dwellings (major + minor)")
df["major_all"].plot(ax=ax[1], color="grey", lw=0.8, label="major total, all uses (old series)")
ax[1].set_ylabel("Permissions granted (decisions)")
ax[1].legend(fontsize=8)
ax[1].set_xlabel(None)
fig.suptitle("England: private starts and granted planning permissions")
plt.tight_layout()
plt.show()

## Step 1 — Integration order

Somerville (2001, p.172) treats permits, starts and completions as *flows* that
describe changes in the housing stock: "the stock of housing also [is]
nonstationary. Since starts, permits, and completions describe changes in the
stock, they should be I(0), which they are in these data." The transform is
therefore chosen from the test rather than imposed: if the England
seasonally-adjusted levels come back I(0), the model is estimated in levels, and
only a confirmed I(1) finding would justify differencing.

ADF and KPSS are run on both the seasonally-adjusted and the raw levels, with
and without a deterministic trend. Seasonal adjustment uses the same
log-additive classical decomposition (centred 2×4 moving average) as
`python/functions/bridge.py`.

Following Somerville's own point on p.172, **no cointegration test is run**:
flow series that are already I(0) "cannot strictly be co-integrated", so a
Johansen or bounds test would be the wrong instrument here.

In [ ]:
def seasonally_adjust(s):
    # log-additive SA, same method as bridge.seasonal_factors
    ln = np.log(s)
    trend = ln.rolling(4, center=True).mean().rolling(2, center=True).mean()
    dev = (ln - trend).dropna()
    f = dev.groupby(dev.index.quarter).mean()
    f = f - f.mean()
    return np.exp(ln - s.index.quarter.map(f).values)


def unit_root_row(name, basis, x, reg):
    adf = adfuller(x, autolag="AIC", regression=reg)
    kp = kpss(x, regression={"c": "c", "ct": "ct"}[reg], nlags="auto")
    return {"series": name, "basis": basis,
            "deterministic": {"c": "constant", "ct": "constant + trend"}[reg],
            "adf_stat": adf[0], "adf_p": adf[1], "adf_lags": adf[2],
            "kpss_stat": kp[0], "kpss_p": kp[1]}


rows = []
for name in ["starts", "granted", "major_all"]:
    for basis, x in [("SA", seasonally_adjust(df[name])), ("NSA", df[name])]:
        for reg in ["c", "ct"]:
            rows.append(unit_root_row(name, basis, x, reg))

ur = pd.DataFrame(rows)
ur.to_csv(f"{OUT}/planning_unit_root_tests.csv", index=False)
ur.round(4)

### Reading the test results

**Private starts (SA levels).** ADF rejects the unit root at 1% with a constant
(−3.61, p = 0.006) and at 5% with constant + trend; KPSS does not reject
stationarity under either. Starts are I(0). This restates what the VECM chapter
already found for log starts.

**Granted residential permissions (SA levels).** ADF does *not* reject with a
constant alone (−1.33, p = 0.62) and KPSS rejects level-stationarity (p ≤ 0.01),
but once a deterministic trend is allowed ADF rejects at 5% (−3.58, p = 0.032)
and KPSS does not reject (p ≥ 0.10). The series is trend-stationary, i.e. I(0)
around a downward deterministic path — not I(1).

**Conclusion.** Both sides of the equation are I(0), exactly as Somerville's
flow-variable argument predicts (2001, p.172), so the model is estimated **in
levels** with seasonal dummies. The previous Δ⁴ transform over-differenced two
stationary series; that is the most likely reason the earlier single-lag
augmentation returned a wrong-signed, insignificant coefficient. Because
permissions are trend- rather than level-stationary, a linear-trend variant of
the final model is reported in §5.

## Step 2 — Estimating equation

Following Somerville (2001, eq. 1, p.166):

$$
\text{starts}_t \;=\; \mu + \sum_{k=0}^{K}\alpha_k\,\text{granted}_{t-k}
\;+\; \sum_{j=1}^{3}\delta_j\,sd_j \;+\; \text{impulse dummies} \;+\; \varepsilon_t
$$

Seasonality is handled by the **centred** quarterly dummies
$sd_j = \mathbb{1}[q=j] - 1/4$, matching `ca.jo(season=4)` and the
ARDL/NARDL convention in `R/05_ARDL.R`, so "SA" in what follows means
regression-adjusted rather than pre-filtered.

Impulse dummies are the subset of the standard five that plausibly hit
permission-to-start timing: `d08Q3`, `d20Q2`, `d20Q3`. `d23Q2`/`d23Q3` are
dropped here — they mark the Part L building-regulations transition, which
shifted the *composition* of starts rather than the permission-to-start
mechanism. They are reinstated as a diagnostic check in §4 if the residuals ask
for them.

In [ ]:
for j in (1, 2, 3):
    df[f"sd{j}"] = (df.index.quarter == j).astype(float) - 0.25

for name, per in [("d08Q3", "2008Q3"), ("d20Q2", "2020Q2"), ("d20Q3", "2020Q3"),
                  ("d23Q2", "2023Q2"), ("d23Q3", "2023Q3")]:
    df[name] = (df.index == per).astype(float)

df["trend"] = np.arange(len(df), dtype=float)

for k in range(KMAX + 1):
    df[f"granted_l{k}"] = df["granted"].shift(k)
    df[f"majall_l{k}"] = df["major_all"].shift(k)
for L in range(1, PMAX + 1):
    df[f"starts_l{L}"] = df["starts"].shift(L)

DET = ["sd1", "sd2", "sd3", "d08Q3", "d20Q2", "d20Q3"]

# Fixed estimation sample across the whole sweep so AIC/BIC stay comparable
EST = df.dropna(subset=[f"granted_l{k}" for k in range(KMAX + 1)]
                       + [f"starts_l{L}" for L in range(1, PMAX + 1)]).copy()
print(f"estimation sample: {EST.index[0]} - {EST.index[-1]}  (n = {len(EST)})")


def fit_hac(cols, data=None, y="starts"):
    data = EST if data is None else data
    X = sm.add_constant(data[cols], has_constant="add")
    return sm.OLS(data[y], X).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})


def lb_p(res, lag):
    return acorr_ljungbox(res.resid, lags=[lag], return_df=True)["lb_pvalue"].iloc[0]

## Step 3 — Distributed-lag sweep, K = 0 … 12

Each candidate includes *all* lags 0…K of granted permissions, so the sweep is
over the length of the distributed lag rather than the position of a single
spike.

Why the window runs to 12 quarters:

- Somerville's quarterly conversion tables (2001, Table 4, p.174) show
  single-family-equivalent permits essentially fully exercised within one
  quarter, but multifamily-equivalent product taking around four quarters to
  reach ~95% cumulative exercise. England's granted-permissions series mixes
  majors and minors, so the outer bound has to sit well beyond one quarter.
- Ball (2011, Table 5, p.14) documents large and persistent heterogeneity in
  permission-grant timing across English LPAs — median 44 weeks in-system plus
  18 weeks to final approval, with the slowest authority roughly four times the
  fastest. That dispersion is upstream of this equation and smears any single
  clean lag.
- Payne, Serin, James & Adams (2019, §3.4, pp.30–34) find build-out and start
  timing are absorption-paced, not capacity-paced — quoting Adams, Leishman &
  Moore (2009): "the speed at which sites are developed is determined by target
  sales, not production efficiency" — and Letwin (2018) reports a median
  15.5-year build-out on large sites with build-out rate not proportional to
  site size (pp.30–31).
- Ball, Cheshire, Hilber & Yu (2024): English LPAs generally require
  construction to start within three years of permission, giving a
  literature-grounded statutory outer bound of 12 quarters.

In [ ]:
def cum_test(res, cols):
    # cumulative sum of a coefficient block, its HAC se, and the joint zero test
    R = np.array([1.0 if n in cols else 0.0 for n in res.params.index])
    tt = res.t_test(R)
    ft = res.f_test(" = 0, ".join(cols) + " = 0")
    return (float(R @ res.params.values), float(np.squeeze(tt.sd)),
            float(np.squeeze(tt.pvalue)), float(ft.fvalue), float(ft.pvalue))


rows = []
for K in range(KMAX + 1):
    gcols = [f"granted_l{k}" for k in range(K + 1)]
    m = fit_hac(gcols + DET)
    cum, cum_se, cum_p, F, Fp = cum_test(m, gcols)
    rows.append({"K": K, "n": int(m.nobs), "adj_r2": m.rsquared_adj,
                 "aic": m.aic, "bic": m.bic,
                 "lb4_p": lb_p(m, 4), "lb8_p": lb_p(m, 8),
                 "cum_coef": cum, "cum_se": cum_se, "cum_p": cum_p,
                 "joint_F": F, "joint_p": Fp})

ksweep = pd.DataFrame(rows).set_index("K")
ksweep.to_csv(f"{OUT}/planning_K_sweep.csv")

print(f"adj-R2 max at K={ksweep.adj_r2.idxmax()} | "
      f"AIC min at K={ksweep.aic.idxmin()} | BIC min at K={ksweep.bic.idxmin()}")
ksweep.round(4)

In [ ]:
# Best *single* lag, judged on its own HAC t-stat -- reported separately from
# the cumulative selection above because the two need not agree.
single = pd.DataFrame([
    {"k": k,
     "coef": (m := fit_hac([f"granted_l{k}"] + DET)).params[f"granted_l{k}"],
     "hac_t": m.tvalues[f"granted_l{k}"],
     "p": m.pvalues[f"granted_l{k}"],
     "adj_r2": m.rsquared_adj}
    for k in range(KMAX + 1)
]).set_index("k")

single.to_csv(f"{OUT}/planning_single_lag.csv")
print(f"strongest single lag: k = {single.hac_t.abs().idxmax()}")
single.round(4)

### Sense-check against the raw data

Somerville's Table 4 (p.174) tracks permit cohorts into starts using micro data.
Nothing equivalent exists for England at national level, so two regression-free
checks stand in:

1. **Raw ratio table** — starts$_t$ / granted$_{t-k}$ for each k, as specified.
2. **Detrended cross-correlation** — correlation of log SA, linearly detrended
   starts with the same transform of permissions at lag k.

The raw ratio is reported first because it was asked for, but it carries almost
no timing information: both series are trending levels, so the ratio is close to
constant in k by construction. The detrended cross-correlation is the check that
actually discriminates between lags, and it is the one compared against K*.

In [ ]:
conv = pd.DataFrame([
    {"k": k,
     "mean_ratio": (r := (df["starts"] / df["granted"].shift(k)).dropna()).mean(),
     "median_ratio": r.median(),
     "p25": r.quantile(.25), "p75": r.quantile(.75), "n": len(r)}
    for k in range(KMAX + 1)
]).set_index("k")


def detrended_log_sa(s):
    x = np.log(seasonally_adjust(s))
    return pd.Series(sm.OLS(x, sm.add_constant(np.arange(len(x), dtype=float))).fit().resid,
                     index=x.index)

S, G = detrended_log_sa(df["starts"]), detrended_log_sa(df["granted"])
conv["ccf_detrended"] = [S.corr(G.shift(k)) for k in range(KMAX + 1)]
conv["ccf_share_of_k0"] = conv["ccf_detrended"] / conv["ccf_detrended"].iloc[0]

conv.to_csv(f"{OUT}/planning_conversion_sensecheck.csv")
conv.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(conv.index, conv["median_ratio"], color="#2c7bb6")
ax[0].set_title("Raw ratio  starts$_t$ / granted$_{t-k}$ (median)")
ax[0].set_xlabel("lag k (quarters)")

ax[1].bar(conv.index, conv["ccf_detrended"], color="#d7191c")
ax[1].axhline(0, color="black", lw=0.6)
ax[1].axhline(1.96 / np.sqrt(len(S)), color="grey", ls="--", lw=0.8)
ax[1].set_title("Cross-correlation, log SA detrended")
ax[1].set_xlabel("lag k (quarters)")
plt.tight_layout()
plt.show()

**Read.** The raw ratio sits at roughly 2.65 dwellings started per residential
permission granted and is flat across k to three decimal places — as expected,
and so uninformative about timing. The detrended cross-correlation decays
monotonically from 0.53 at k = 0, is still outside the ±1.96/√n band at k = 6,
and crosses zero around k = 10. Most of the co-movement is concentrated in the
first four to five quarters, so the profile brackets the regression-selected
K* (below) rather than contradicting it — if anything the raw check tolerates a
slightly longer lag than adjusted R²/AIC pick. No material divergence to flag.

## Step 4 — Own-lag sweep and final estimation

K* is fixed at the value selected in Step 3 and starts' own lags are swept over
1…4, the same range as the completions-bridge protocol, on the same fixed
sample. Selection is by adjusted R² / AIC / BIC together with Ljung-Box on the
residuals — an own-lag order that leaves serial correlation is rejected however
well it scores on fit.

In [ ]:
KSTAR = int(ksweep.adj_r2.idxmax())
GCOLS = [f"granted_l{k}" for k in range(KSTAR + 1)]
print(f"K* = {KSTAR}")

rows = []
for P in range(PMAX + 1):
    scols = [f"starts_l{L}" for L in range(1, P + 1)]
    m = fit_hac(GCOLS + scols + DET)
    cum, cum_se, cum_p, F, Fp = cum_test(m, GCOLS)
    rho = sum(m.params[c] for c in scols)
    rows.append({"P": P, "adj_r2": m.rsquared_adj, "aic": m.aic, "bic": m.bic,
                 "lb4_p": lb_p(m, 4), "lb8_p": lb_p(m, 8),
                 "cum_coef": cum, "cum_p": cum_p, "joint_F": F, "joint_p": Fp,
                 "sum_rho": rho, "long_run": cum / (1 - rho)})

psweep = pd.DataFrame(rows).set_index("P")
psweep.to_csv(f"{OUT}/planning_ownlag_sweep.csv")
print(f"adj-R2 max at P={psweep.adj_r2.idxmax()} | "
      f"AIC min at P={psweep.aic.idxmin()} | BIC min at P={psweep.bic.idxmin()}")
psweep.round(4)

Adjusted R² and AIC both pick P = 4; BIC prefers the more parsimonious P = 2.
Ljung-Box settles it — P = 2 leaves residual autocorrelation at lag 4
(p = 0.037) while P = 4 is clean at both lag 4 and lag 8. Whiteness takes
precedence over BIC parsimony here, the same call made in the completions
bridge, so the headline model is K* = 4, P* = 4.

In [ ]:
PSTAR = int(psweep.adj_r2.idxmax())
SCOLS = [f"starts_l{L}" for L in range(1, PSTAR + 1)]
final = fit_hac(GCOLS + SCOLS + DET)
print(final.summary())

In [ ]:
cum, cum_se, cum_p, F, Fp = cum_test(final, GCOLS)
rho = sum(final.params[c] for c in SCOLS)

# Delta-method SE for the long-run multiplier sum(alpha) / (1 - sum(rho))
R_a = np.array([1.0 if n in GCOLS else 0.0 for n in final.params.index])
R_r = np.array([1.0 if n in SCOLS else 0.0 for n in final.params.index])
lr = cum / (1 - rho)
grad = R_a / (1 - rho) + (cum / (1 - rho) ** 2) * R_r
lr_se = float(np.sqrt(grad @ final.cov_params().values @ grad))

bg_stat, bg_p = acorr_breusch_godfrey(
    sm.OLS(EST["starts"], sm.add_constant(EST[GCOLS + SCOLS + DET])).fit(), nlags=4)[2:]

# Do the dropped regulatory dummies want back in?
with_regstd = fit_hac(GCOLS + SCOLS + DET + ["d23Q2", "d23Q3"])
regstd_F = with_regstd.f_test("d23Q2 = 0, d23Q3 = 0")

diag = {
    "K*": KSTAR, "P*": PSTAR, "n": int(final.nobs),
    "adj_r2": final.rsquared_adj, "aic": final.aic, "bic": final.bic,
    "ljung_box_4_p": lb_p(final, 4), "ljung_box_8_p": lb_p(final, 8),
    "breusch_godfrey_4_stat": bg_stat, "breusch_godfrey_4_p": bg_p,
    "cum_granted": cum, "cum_granted_se": cum_se, "cum_granted_p": cum_p,
    "joint_F": F, "joint_p": Fp,
    "sum_rho": rho, "long_run_mult": lr, "long_run_se": lr_se,
    "regstd_joint_F": float(regstd_F.fvalue), "regstd_joint_p": float(regstd_F.pvalue),
}
pd.Series(diag).to_csv(f"{OUT}/planning_final_diagnostics.csv", header=False)

for k, v in diag.items():
    print(f"{k:24s} {v:>12.4f}")

### Diagnostics

- **Residual whiteness.** Ljung-Box p = 0.372 at lag 4 and 0.412 at lag 8 —
  clean. Breusch-Godfrey at lag 4 is marginal (p = 0.038). BG is run on the
  OLS-covariance fit, so it does not carry the HAC correction the reported
  standard errors do; the residual dependence it picks up is what
  Newey-West(4) is already handling. This is recorded as a shared limitation
  with the ARDL/VECM chapters rather than chased with further respecification.
- **Regulatory dummies.** Adding `d23Q2`/`d23Q3` back is not neutral: they are
  jointly significant (F = 174.2) and lift adjusted R² to 0.865, because two
  point dummies absorb the very large 2023 pull-forward-and-collapse in starts
  almost exactly. But their inclusion *breaks* residual whiteness
  (Ljung-Box p = 0.025 at lag 4, 0.054 at lag 8) while the model without them
  is clean at both. Since the selection protocol here puts whiteness ahead of
  fit, they stay out — which also matches the substantive reading that the
  Part L transition shifted the composition of starts rather than the
  permission-to-start mechanism. The tension is recorded rather than resolved:
  two point dummies on two observations are a low-power basis for either call,
  the same caveat made in `completions_spec_comparison.ipynb`.
- **Normality.** Jarque-Bera rejects heavily (kurtosis ≈ 13.7), driven by the
  2020 quarters. HAC inference does not need normal errors, but it means the
  reported intervals are asymptotic.
- **Cumulative effect.** The five granted-permission lags are *jointly* highly
  significant (F = 10.28, p < 0.001), but their *sum* is not individually
  distinguishable from zero (Σα = 0.229, HAC se = 0.217, p = 0.29). The implied
  long-run multiplier is 0.95 dwellings started per permission granted, with a
  delta-method standard error of 0.64. Permissions clearly carry information
  about starts; the magnitude of the pass-through is not pinned down. Both
  numbers are reported in §6 — the joint test alone would overstate the
  precision of the result.

In [ ]:
def coef_table(res, order=None):
    t = pd.DataFrame({"coef": res.params, "hac_se": res.bse,
                      "t": res.tvalues, "p": res.pvalues})
    return t.loc[order] if order else t


LABELS = {"const": "Constant", **{f"granted_l{k}": rf"Granted$_{{t-{k}}}$" for k in range(KMAX + 1)},
          **{f"starts_l{L}": rf"Starts$_{{t-{L}}}$" for L in range(1, PMAX + 1)},
          "sd1": "Seasonal Q1", "sd2": "Seasonal Q2", "sd3": "Seasonal Q3",
          "d08Q3": "2008Q3 (GFC)", "d20Q2": "2020Q2 (Covid)", "d20Q3": "2020Q3 (Covid)",
          "trend": "Trend"}


def to_booktabs(tbl, caption, label, notes, path):
    stars = lambda p: "***" if p < .01 else "**" if p < .05 else "*" if p < .1 else ""
    lines = [r"\begin{table}[htbp]", r"\centering",
             rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{lrr}", r"\toprule",
             r"& Coefficient & HAC s.e. \\", r"\midrule"]
    for name, r in tbl.iterrows():
        lines.append(rf"{LABELS.get(name, name)} & {r['coef']:.3f}{stars(r['p'])} "
                     rf"& ({r['hac_se']:.3f}) \\")
    lines += [r"\bottomrule", r"\end{tabular}",
              r"\begin{minipage}{\linewidth}\vspace{4pt}\footnotesize " + notes + r"\end{minipage}",
              r"\end{table}"]
    tex = "\n".join(lines)
    with open(path, "w") as fh:
        fh.write(tex)
    return tex


ORDER = ["const"] + GCOLS + SCOLS + DET
ctab = coef_table(final, ORDER)
ctab.to_csv(f"{OUT}/planning_final_coefficients.csv")

notes = (rf"Dependent variable: England private-enterprise housing starts, {EST.index[0]}--{EST.index[-1]} "
         rf"($n={int(final.nobs)}$). Newey--West HAC standard errors, {HAC_LAGS} lags. "
         rf"Seasonal dummies are centred. Adjusted $R^2 = {final.rsquared_adj:.3f}$; "
         rf"Ljung--Box $p = {lb_p(final, 4):.3f}$ (lag 4), ${lb_p(final, 8):.3f}$ (lag 8). "
         rf"Cumulative permissions effect $\sum\alpha_k = {cum:.3f}$ (s.e.\ {cum_se:.3f}), "
         rf"joint $F = {F:.2f}$, $p < 0.001$; implied long-run multiplier "
         rf"${lr:.3f}$ (s.e.\ {lr_se:.3f}). $^{{*}}p<0.1$, $^{{**}}p<0.05$, $^{{***}}p<0.01$.")

print(to_booktabs(ctab, "Granted planning permissions and private housing starts, England",
                  "tab:planning_starts", notes, f"{OUT}/planning_final_coefficients.tex"))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
b = final.params[GCOLS].values
se = final.bse[GCOLS].values
ax.errorbar(range(KSTAR + 1), b, yerr=1.96 * se, fmt="o", capsize=4, color="#2c7bb6")
ax.axhline(0, color="black", lw=0.6)
ax.set_xlabel("lag k (quarters)")
ax.set_ylabel("Starts per permission granted")
ax.set_title(f"Distributed-lag profile (K* = {KSTAR}, P* = {PSTAR}), 95% HAC intervals")
plt.tight_layout()
plt.show()

## Step 5 — Robustness

Separate from the headline specification. The question is whether the
permissions→starts relationship survives controls, not whether a better-fitting
model exists.

Somerville (2001) is explicit that price and volatility effects on starts are
small and fragile in this literature: elasticities of roughly 0.007
(single-family) and 0.096 (multifamily) (p.163), and no robust real-options or
conditional-volatility effect for either starts or completions (pp.178–179).
His price term is instrumented (lagged population and employment growth,
mortgage rates); no comparable instrument set is readily available in this
dataset, so real house-price growth enters directly. **That is an endogeneity
limitation, not an identified price effect**, and the coefficient is not
interpreted structurally.

The absorption/demand proxy is YoY growth in transaction volumes — the closest
thing in `england_master.csv` to the sales-rate channel Payne et al. (2019)
identify. Volatility is an 8-quarter rolling standard deviation of quarterly
real house-price growth, a deliberately simple stand-in for Somerville's GARCH
conditional variance.

Also reported here: the linear-trend variant flagged in §1, the original
`major total (all)` permissions series, and three subsamples.

In [ ]:
df["dlrprc"] = np.log(df["hprice"] / df["gdp_def"]).diff(4)
df["dlvol"] = np.log(df["vol"]).diff(4)
df["hp_volat"] = np.log(df["hprice"] / df["gdp_def"]).diff().rolling(8).std()

EST2 = df.loc[EST.index].dropna(subset=["dlrprc", "dlvol", "hp_volat"])


def rob_row(name, cols, data=None, gcols=None):
    gcols = gcols or GCOLS
    m = fit_hac(cols, data)
    cum, cum_se, cum_p, F, Fp = cum_test(m, gcols)
    rho = sum(m.params[c] for c in SCOLS)
    return {"spec": name, "n": int(m.nobs), "cum_coef": cum, "cum_se": cum_se,
            "cum_p": cum_p, "joint_F": F, "joint_p": Fp,
            "long_run": cum / (1 - rho), "adj_r2": m.rsquared_adj, "lb4_p": lb_p(m, 4)}


BASE = GCOLS + SCOLS + DET
GA = [f"majall_l{k}" for k in range(KSTAR + 1)]

rob = pd.DataFrame([
    rob_row("headline", BASE),
    rob_row("+ linear trend", BASE + ["trend"]),
    rob_row("+ real house-price growth", BASE + ["dlrprc"], EST2),
    rob_row("+ transactions growth", BASE + ["dlvol"], EST2),
    rob_row("+ house-price volatility", BASE + ["hp_volat"], EST2),
    rob_row("+ all three controls", BASE + ["dlrprc", "dlvol", "hp_volat"], EST2),
    rob_row("old series: major total (all)", GA + SCOLS + DET, gcols=GA),
    rob_row("sample 1982Q2-2007Q4", BASE, EST.loc[:"2007Q4"]),
    rob_row("sample 1997Q1-2025Q4", BASE, EST.loc["1997Q1":]),
    rob_row("sample 1982Q2-2019Q4 (pre-Covid)", BASE, EST.loc[:"2019Q4"]),
]).set_index("spec")

rob.to_csv(f"{OUT}/planning_robustness.csv")
rob.round(4)

In [ ]:
controls = fit_hac(BASE + ["dlrprc", "dlvol", "hp_volat"], EST2)
coef_table(controls, ["dlrprc", "dlvol", "hp_volat"]).round(4)

**Read.**

- The cumulative permissions effect is stable across every control set
  (0.21–0.28) and jointly significant throughout (p < 0.001). Adding a linear
  trend pulls it to 0.16 but leaves the joint test untouched, so the
  trend-stationarity of permissions is not driving the result.
- **Real house-price growth is insignificant** (HAC t = 1.03) and
  **volatility is insignificant** (t = −0.56). Both reproduce Somerville's own
  finding of small, fragile price effects and no robust real-options channel
  (pp.163, 178–179) — with the caveat that the price term here is not
  instrumented and so is not an identified elasticity.
- **Transactions growth is significant** (t = 2.33) and raises adjusted R² more
  than either other control, consistent with the absorption-paced view in Payne
  et al. (2019, §3.4).
- **The old `major total (all)` series produces a wrong-signed cumulative
  coefficient** (−0.204). That series is dominated by non-residential majors,
  and it is the reason the previous version of this notebook returned a negative
  permissions effect. The data-construction correction in §1 is doing real work.
- **Subsample instability is substantial**: the implied long-run multiplier is
  0.28 in 1982–2007, 1.82 in 1997–2025, 0.73 pre-Covid. The point estimate is
  not stable across regimes, which reinforces the imprecision already visible in
  the full-sample standard error.

## Step 6 — Interpretation

**1. Integration order.** England's seasonally-adjusted private starts and
granted residential permissions are both I(0) in levels — starts around a
constant, permissions around a deterministic trend. This is exactly what
Somerville (2001, p.172) argues should hold for flow variables: because starts
and permits describe *changes* in a nonstationary housing stock, they should
themselves be I(0), and are. The model is therefore estimated in levels with
centred seasonal dummies. The previous Δ⁴ specification differenced two already
stationary series, discarding the level information the relationship lives in.
No cointegration test is run, per Somerville's point that I(0) flows "cannot
strictly be co-integrated".

**2. Selected lag length.** Adjusted R² and AIC both select **K* = 4** quarters
of granted permissions (BIC prefers K = 3, a one-quarter difference).
Starts' own lags are set at **P* = 4** on Ljung-Box grounds. The detrended
cross-correlation sense-check decays monotonically from 0.53 at k = 0, remains
outside the ±1.96/√n band through k = 6, and crosses zero near k = 10 — so the raw-data timing profile brackets K* rather than contradicting
it. The strongest *single* lag is k = 0 (HAC t = 5.86), which differs from the
cumulative selection, as anticipated. The flat raw ratio table (≈2.65 across all
k) carries no timing content and is reported only for transparency.

**3. Cumulative effect.** The five permission lags are jointly highly
significant: F = 10.28, p < 0.001. Their sum is **Σα = 0.229 dwellings started
per residential permission granted** over the four quarters following the grant
(HAC s.e. 0.217), implying a **long-run multiplier of 0.95** (delta-method s.e.
0.64) once the own-lag dynamics are unwound. The honest statement is therefore
two-sided: permissions carry clear joint explanatory power for starts, and the
model is a substantial improvement on the previous specification (adjusted
R² 0.759 against 0.363), **but the size of the pass-through is not precisely
estimated** — the sum is not individually distinguishable from zero at
conventional levels, and the subsample estimates in §5 range from 0.28 to 1.82.
It is not claimed here that a granted permission mechanically delivers a start.

**4. Limitation — and where this stops.** Two mechanisms sit between a
permission and a start, both outside this equation:

- *Upstream aggregation noise.* Ball (2011, Table 5, p.14) documents large and
  persistent local-authority heterogeneity in permission-grant timing itself:
  a median of 44 weeks in-system plus 18 weeks to final approval, with the
  slowest authority around four times the fastest. Aggregating 300-plus LPAs
  with dispersed and non-stationary processing times into one England quarterly
  series smears the timing signal before a start can occur. Part of the
  imprecision above is baked in at the data level, not evidence of no
  relationship.
- *Downstream absorption pacing.* Payne, Serin, James & Adams (2019, §3.4,
  pp.30–34) and Letwin (2018) show start and build-out timing are set by
  housebuilders' own sales-rate targets, not triggered mechanically by the
  grant of permission — "the speed at which sites are developed is determined
  by target sales, not production efficiency" (Adams, Leishman & Moore 2009,
  quoted p.32). The significant transactions-growth control in §5 is consistent
  with that channel. Letwin's median 15.5-year build-out on large sites, and his
  finding that build-out rate is not proportional to site size, mean the
  three-year statutory commencement window bounds when a start *may* occur but
  not when it *will*.

A third, measurement-level caveat: PS2 counts decisions rather than dwellings,
so the mix of scheme sizes behind one "permission" varies over the sample. The
coefficients are dwellings-per-permission, not a unit conversion rate.

Taken together, these are reasons to read the cumulative estimate as a lower
bound on precision rather than a settled magnitude. **This is stated as a
limitation and the specification search stops here** — consistent with the
under-claiming standard applied in the ARDL, NARDL and VECM chapters.